# 06 — Submission public test trên Google Colab (ASR fine-tune fp32 + Qwen3-4B bf16)

**Runtime → Change runtime type → GPU: L4 (24 GB, có bf16) hoặc A100.** T4 (15 GB, không bf16) dễ OOM/timeout.

Chuẩn bị trên Google Drive của bạn (My Drive):
1. Shortcut thư mục `ZaloAIC_LifeMate_handoff` (được share) → *Organize → Add shortcut → My Drive*. Trong đó có `phowhisper_medium_ft_v1_fp32.zip` (+ `.sha256`).
2. File dữ liệu BTC `train+public_test.zip` — tải từ portal bằng tài khoản đội, đặt ở My Drive (sửa `DATA_ZIP` nếu để chỗ khác).

Chạy lần lượt từng ô (hoặc Runtime → Run all). Kết quả ghi thẳng vào Drive (`OUT_DIR`) để không mất khi Colab ngắt.

In [ ]:
# ===== Cấu hình =====
HANDOFF = "/content/drive/MyDrive/ZaloAIC_LifeMate_handoff"
DATA_ZIP = "/content/drive/MyDrive/train+public_test.zip"
OUT_DIR = "/content/drive/MyDrive/lifemate_out"
REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/Zalo-AIC-2026-Lifemate.git"
BRANCH = "feat/asr-ft-native-dtype"
LLM_MODEL = "Qwen/Qwen3-4B"
RUN_DEV_CHECK = True      # 48 mẫu pilot dev có nhãn trước public test (vài chục phút) — để biết điểm/timeout sớm
RUN_NAME = "public_asr_ft_v1"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import torch, subprocess
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime -> Change runtime type"
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory/2**30:.1f} GB | bf16 phần cứng: {p.major >= 8}")
if p.total_memory < 20 * 2**30:
    print("CẢNH BÁO: GPU < 20 GB — ASR fp32 + Qwen bf16 có thể không đủ VRAM; nên chọn L4 hoặc A100.")

In [ ]:
# ===== Source (nhánh có bản sửa dtype gốc) + thư viện =====
!rm -rf /content/src_repo && git clone -q --depth 1 -b {BRANCH} {REPO_URL} /content/src_repo
!pip install -q "transformers>=4.51.0,<5" "accelerate>=1.0,<2" "safetensors>=0.5,<1"
src = open("/content/src_repo/src/lifemate/models.py", encoding="utf-8").read()
assert 'torch_dtype="auto"' in src, "Source chưa có bản sửa dtype — kiểm tra BRANCH"
print("source OK")

In [ ]:
# ===== Model ASR + dữ liệu: chép từ Drive sang ổ /content (đọc từ Drive mount rất chậm) =====
import os, hashlib, zipfile, shutil
from pathlib import Path
os.makedirs("/content/asr", exist_ok=True); os.makedirs("/content/data", exist_ok=True)
zp = "/content/asr/model.zip"
shutil.copy(f"{HANDOFF}/phowhisper_medium_ft_v1_fp32.zip", zp)
h = hashlib.sha256()
with open(zp, "rb") as f:
    for b in iter(lambda: f.read(1 << 24), b""): h.update(b)
expect = open(f"{HANDOFF}/phowhisper_medium_ft_v1_fp32.zip.sha256").read().split()[0]
assert h.hexdigest() == expect, "Sai checksum model — tải lại"
zipfile.ZipFile(zp).extractall("/content/asr"); os.remove(zp)
ASR_MODEL = "/content/asr/phowhisper_medium_ft_v1_fp32"

shutil.copy(DATA_ZIP, "/content/data/data.zip")
zipfile.ZipFile("/content/data/data.zip").extractall("/content/data"); os.remove("/content/data/data.zip")
hits = [p.parent.parent for p in Path("/content/data").rglob("train/train.jsonl")
        if (p.parent.parent / "public_test/public_test.jsonl").is_file()]
assert len(hits) == 1, hits
DATA_ROOT = hits[0]
print("ASR:", ASR_MODEL, "\nDATA:", DATA_ROOT)

In [ ]:
# ===== Kiểm tra nhanh ASR (fp32) bằng class của team =====
import sys, json, time
sys.path.insert(0, "/content/src_repo/src")
from lifemate.models import PhoWhisper
row = json.loads(open(DATA_ROOT / "train/train.jsonl", encoding="utf-8").readline())
m = PhoWhisper(ASR_MODEL, "cuda:0")
assert m.model.dtype == torch.float32, m.model.dtype
t0 = time.time(); hyp = m.transcribe(DATA_ROOT / "train" / row["audio"], 30)
print(f"dtype {m.model.dtype} | {time.time()-t0:.1f}s\nGT : {row['audio_text']}\nASR: {hyp}")
del m; torch.cuda.empty_cache()

In [ ]:
# ===== Hàm chạy module của team + đo đỉnh VRAM =====
import threading
ENV = dict(os.environ, PYTHONPATH="/content/src_repo/src", PYTHONHASHSEED="42")
PEAK = {"mb": 0}
def _watch(stop):
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.split()
        if out: PEAK["mb"] = max(PEAK["mb"], int(out[0]))
        stop.wait(2)

def run_module(module, *args):
    stop = threading.Event(); t = threading.Thread(target=_watch, args=(stop,), daemon=True); t.start()
    try:
        subprocess.run([sys.executable, "-m", "lifemate." + module, *map(str, args)], env=ENV, check=True)
    finally:
        stop.set(); t.join()
        if module == "run": print("Đỉnh VRAM:", PEAK["mb"], "MB")

DEVICES = ["--asr-device", "cuda:0", "--llm-device", "cuda:0"]
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ===== (Tùy chọn) Pilot 48 mẫu dev có nhãn =====
if RUN_DEV_CHECK:
    AUDIT = Path("/content/work/data_audit")
    if not (AUDIT / "pilot_ids.json").exists():
        run_module("audit", "--data-root", DATA_ROOT, "--output", AUDIT)
    PILOT = Path(OUT_DIR) / "pilot_asr_ft_v1"
    if not (PILOT / "evaluation.json").exists():
        shutil.rmtree(PILOT, ignore_errors=True)
        run_module("run", "--data-root", DATA_ROOT, "--split", "train", "--ids", AUDIT / "pilot_ids.json",
                   "--asr-model", ASR_MODEL, "--llm-model", LLM_MODEL, *DEVICES, "--output", PILOT)
        run_module("evaluate", "--data-root", DATA_ROOT, "--run", PILOT)
    r = json.loads((PILOT / "evaluation.json").read_text())
    print(json.dumps({k: r.get(k) for k in ["samples", "state_match_rate", "mean_exact_call_f1", "asr",
                                             "latency_seconds", "stop_reasons"]}, ensure_ascii=False, indent=1))

In [ ]:
# ===== Public test -> submission.jsonl (ghi thẳng vào Drive) =====
PUBLIC_RUN = Path(OUT_DIR) / RUN_NAME
assert not (PUBLIC_RUN / "submission.jsonl").exists() or input("Đã có submission. Gõ y để chạy lại: ") == "y"
shutil.rmtree(PUBLIC_RUN, ignore_errors=True)     # run.py cần thư mục output MỚI
t0 = time.time()
run_module("run", "--data-root", DATA_ROOT, "--split", "public_test",
           "--asr-model", ASR_MODEL, "--llm-model", LLM_MODEL, *DEVICES, "--output", PUBLIC_RUN)
print(f"public run: {(time.time()-t0)/60:.1f} phút")

In [ ]:
# ===== Kiểm tra định dạng bằng công cụ BTC + đủ id =====
subprocess.run([sys.executable, "-m", "zaic_tools", "--split", "public_test", "--root", str(DATA_ROOT),
                "--validate", str(PUBLIC_RUN / "submission.jsonl")],
               env=dict(ENV, PYTHONPATH=str(DATA_ROOT / "public_test")), check=True)
rows = [json.loads(l) for l in (PUBLIC_RUN / "submission.jsonl").read_text(encoding="utf-8").splitlines()]
expected = [json.loads(l)["id"] for l in (DATA_ROOT / "public_test/public_test.jsonl").read_text(encoding="utf-8").splitlines()]
assert len(rows) == len(expected) and {r["id"] for r in rows} == set(expected), (len(rows), len(expected))
print(f"OK: {len(rows)} dòng. File nộp (trên Drive): {PUBLIC_RUN / 'submission.jsonl'}")